# Adding adsorbates on surfaces with pySNOW

Surfaces of nanoparticles and other atomistic systems are central in catalysis, as they are the sites where atoms and molecules are adsorbed, possibly changing very significantly the reaction paths, their activation energies, and the course of the reaction in general. Here, we show how to use pySNOW (and specifically the `add_molecule` module) to add adsorbed molecule on an atomistic configuration.

We will use the functions present in `snow.descriptors.coordination` to find relevant surface sites (i.e. atop, bridge, three-hollow, and four-hollow), and subsequently we will add molecules in these sites. In order to do this, we need to define a direction along which to place the molecule - generally, a direction normal to the local geometry of the surface. While this is straightforward for atomic triplets (which inherently define a plane), it needs a bit of discussion for atop and bridge sites. 

The algorithm we developed considers the local environment of the provided atop or bridge site (which includes all atoms closer to the site than a cutoff distance $r_{cut}$), computes its geometrical center of mass, and provides a 'locally normal' direction as the vector connecting the geometric center of mass and the initial site. We have found that a cutoff that includes nearest neighbours works fine, altough you might want to increase it a bit (say, to second nearest neighbours) for some environments. If resulting directions do not come out as expected, try playing with this cutoff a bit.

Finally, for fourplets, we obtain the normal to the surface as the average of normals of all triplets present in the fourplet (which is robust for distorted fourplets where not all atoms re in the same plane).

To check out how do this methods work in practice, their dependence on the cutoff radius of the local environment, and what are the computed locally normal directions on your own files, you can use the `directions.py` code in the `tutorial/` folder. This takes as input a `*.xyz` configuration and a value for the cutoff radius, and writes a `*_directions.xyz` file which contains the normal directions stored as forces for all computed surface sites. These can be visualized, e.g., with ovito.

`python distributions.py <your_config.xyz> <cutoff_radius>`

(notice that, in some cases, it is better to distinguish the local environment cutoff radius from the coordination number calculation radius - namely, every time you increase your environment cutoff above the usual nearest neighbour cutoff $\approx 0.88\ a_{lat}$, otherwise your cordination calculations and subsequent filtering of surface atoms wrt a threshold will be wrong).

In [46]:
#import relevant modules
from snow.catalysis.add_molecule import add_molecule, locally_normal_direction, triplet_normal, fourplet_normal, check_overlapping
from snow.descriptors.coordination import coordination_number, agcn_calculator, bridge_gcn, three_hollow_gcn, four_hollow_gcn #used to find surface sites.
from snow.io.xyz import read_xyz, write_xyz

import numpy as np

In [47]:
#read an initial configuration
el, coords = read_xyz('tutorial_structures/Cu247_Dh.xyz') #an elongated decahedron

#define some adatoms and molecules of increasing comlexity you can place on your surfaces.

el_adatom, coords_adatom = ['Cu'], np.array([[0.,0.,0.]]) #single Cu atom

el_o2, coords_o2 = ['O', 'O'], np.array([[0.,0.,0.],[0., 0., 1.2]]) #O2 molecule

el_benzene, coords_benzene = read_xyz('tutorial_structures/benzene.xyz') #benzene molecule

The atom or molecule to be added to our intial system will be taken as it is, aligned to a given direction, displaced of a given distance from the given site along the given direction, and finally, possibly rotated to allow for different adsorption geometries. Given this procedure, you should provide your molecules with the anchor atom in the origin of your reference system. If this is not the case, you can resort to this standard by adding a line such as `coords_molecule -= coords_molecule[anchor_id]`, with `anchor_id` being the id of the anchor atom in the el list and coords array.

Let's compute the bridge, three-hollow, and four-hollow sites on the nanoparticle:

In [48]:
alat   = 3.615 # copper lattice constant
cutoff = 0.88*alat #a general good cutoff, between the first and second peak of the pddf
thr_cn = 10 #coordination number threshold to distinguish surface atoms from deeper atoms

#atop sites are simply the coordinates of the atoms in the system, but we can filter out atoms that are not on the surface with a condition on the coordination number
cns = coordination_number(coords,cutoff)
atops = []
for cn, coord in zip(cns, coords):
    if cn<thr_cn:
        atops.append(coord)
atops = np.asarray(atops)

#bridge
b_sites, pairs, bgcns = bridge_gcn(coords, cutoff, thr_cn)

#3hollow
t_sites, triplets, tgcns = three_hollow_gcn(coords, cutoff, thr_cn)

#4hollow
#f_sites, fourplets, fgcns = four_hollow_gcn(coords, cutoff, thr_cn) #strangely not working in jupyter notebook - works from commandline or programs...

[==================================================] 100.00%

Now that we have sites, let's add structures.

In [49]:
#Copper adatom
site_id   = 20
site      = atops[site_id]
distance  = 2.556 #Cu bulk nn distance 

#compute the locally normal direction
direction = locally_normal_direction(coords, site, cutoff) #keeping the same cutoff as before - taking into account nearest neighbours

#add the molecule
el_with_adatom, coords_with_adatom = add_molecule(el, coords, site, direction, distance, el_adatom, coords_adatom)

# print(len(el))
# print(coords.shape)
# print(len(el_adatom))
# print(coords_adatom.shape)
# print(len(el_with_adatom))
# print(coords_with_adatom.shape)

#optional visualize with ase.view
from ase.visualize import view
from ase import Atoms

atoms = Atoms(symbols = el_with_adatom, positions = coords_with_adatom)
view(atoms, viewer='x3d')

#optional: save the configuration
#write_xyz('Cu247_Dh_adatom.xyz', el, coords)

![Cu adatom](./adatom.png "adatom")

In [50]:
#O2 moelcule
site_id = 9
site = t_sites[site_id]
triplet = triplets[site_id]
distance = 1.88

#compute the locally normal direction
direction = triplet_normal(coords, triplet, cutoff)

#add the molecule
el_with_o2, coords_with_o2 = add_molecule(el, coords, site, direction, distance, el_o2, coords_o2)

#optional visualize with ase.view
from ase.visualize import view
from ase import Atoms

atoms = Atoms(symbols = el_with_o2, positions = coords_with_o2)
view(atoms, viewer='x3d')

![O2](./oxygen.png "o2")

In [38]:
#benzene ring
site = b_sites[50]
distance = 1.81

#compute the locally normal direction
direction = locally_normal_direction(coords, site, cutoff)

#add the molecule
el_with_benzene, coords_with_benzene = add_molecule(el, coords, site, direction, distance, el_benzene, coords_benzene)

#add another one, play with it by tilting it of various angles
theta = np.pi/4  #angle wrt the provided direction
phi   = np.pi/3. #angle around the provided direction

site_2 = b_sites[45]
direction_2 = locally_normal_direction(coords, site_2, cutoff)

el_with_benzene, coords_with_benzene = add_molecule(el_with_benzene, coords_with_benzene, site_2, direction_2, distance, el_benzene, coords_benzene, theta, phi) #extra atoms are simply appended to the symbols list and coordinates array

#optional visualize with ase.view
from ase.visualize import view
from ase import Atoms

atoms = Atoms(symbols = el_with_benzene, positions = coords_with_benzene)
view(atoms, viewer='x3d')

![benzenes](./benzenes.png "benzene")

A final example: let's add a several molecules on a nanoparticle and check that they are not compenetrating.

In [39]:
alat   = 3.57
cutoff = 0.88*alat
distance = 1.82

#a dictionary of atomic radii, to check whether atoms are overlapping
radii_dict = {'Cu': 1.275,
              'C': 0.65,
              'H': 0.39}

print(check_overlapping(el_benzene, coords_benzene, radii_dict))
print(check_overlapping(el, coords, radii_dict))


el, coords = read_xyz('tutorial_structures/Cu309_Ih.xyz')
_, agcns   = agcn_calculator(coords, cutoff)
atops      = np.asarray([ coord for coord, gcn in zip(coords, agcns) if gcn<thr_cn ])

#we will use atop sites, and place benzene molecules on them.
#we sort the sites randomly and add molecules whenever we can find space for them.

indexes = np.random.permutation(len(atops))
overlap_count = 0

for i in indexes:
    site = atops[i]
    direction = locally_normal_direction(coords, site, cutoff)
    theta = np.random.rand()*np.pi/3 #random orientation
    phi   = np.random.rand()*2*np.pi #       " 
    new_el, new_coords = add_molecule(el, coords, site, direction, distance, el_benzene, coords_benzene, theta,phi)

    if not check_overlapping(new_el, new_coords, radii_dict): #check: if there are no overlapping atoms, we accept the new configuration and set it as the new one
        el = new_el
        coords = new_coords
    else:
        #print(f'overlapping config at index {i}')
        overlap_count +=1


print(overlap_count, 'overlapping tries ')
atoms = Atoms(symbols = el, positions = coords)
view(atoms, viewer='x3d')

False
False
92 overlapping tries 


![covered np](./full_cover.png "covered")

This procedure was also put in a function which is part of the `add_molecule` module for convenience. You can find a use case here below:

In [40]:
from snow.catalysis.add_molecule import cover_surface
el_to_cover, coords_to_cover = read_xyz('tutorial_structures/Cu309_Ih.xyz')

alat = 3.57
cutoff = 0.88*alat
thr_cn = 10

el_mol = el_benzene
coords_mol = coords_benzene

distance = 1.9
ratio = 0.9 #how many molecules should I keep out of all the possible ones I can place?
#notice that the 'possible ones' are just the ones I can place with the given angles and filling
#sites in random order, so this is not necessarily the closest packing one can get.
sites_type = 'atop'

theta = 0#np.pi/2
phi = 0.

radii = {'Cu': 1.275,
              'C': 0.65,
              'H': 0.39,
              'O': 0.59}

el_covered, coords_covered = cover_surface(el_to_cover, coords_to_cover, cutoff, thr_cn, el_mol, coords_mol, distance, radii, ratio, sites_type, theta, phi)

atoms = Atoms(symbols = el_covered, positions = coords_covered)
view(atoms, viewer='x3d')

tries that resulted in overlapping atoms: 52
only kept 99 adsorbates out of 110 possible ones 
